# Category 7 - TTS Voice Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares voice-output architectures for Ally Vision v2. The core question is whether speech is a second service call after text generation, or a native part of the same realtime interaction loop.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
source_urls = {
  "Qwen pricing": "https://docs.qwencloud.com/developer-guides/getting-started/pricing",
  "OpenAI Realtime API": "https://openai.com/index/introducing-the-realtime-api/",
  "Google Cloud TTS docs": "https://cloud.google.com/text-to-speech/docs",
  "Azure Speech support": "https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt",
  "ElevenLabs pricing": "https://elevenlabs.io/pricing",
  "Ally realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py"
}

# Hardcoded public-source values only. No runtime web calls are performed in this notebook.


In [ ]:
comparison_rows = [
  {
    "Metric": "Integrated speech output",
    "Qwen Omni TTS (Tina)": "Yes, output is part of the realtime model session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "OpenAI TTS-1 / TTS-1-HD": "Separate TTS layer from general model flow [source: https://openai.com/index/introducing-the-realtime-api/]",
    "Google Cloud TTS WaveNet": "Separate service/API [source: https://cloud.google.com/text-to-speech/docs]",
    "ElevenLabs Streaming": "Separate service/API [source: https://elevenlabs.io/pricing]",
    "Azure Neural TTS": "Separate service/API [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]"
  },
  {
    "Metric": "Streaming / realtime speech",
    "Qwen Omni TTS (Tina)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "OpenAI TTS-1 / TTS-1-HD": "OpenAI article contrasts stitched TTS with realtime API path [source: https://openai.com/index/introducing-the-realtime-api/]",
    "Google Cloud TTS WaveNet": "Streaming/bidirectional TTS documented in product family [source: https://cloud.google.com/text-to-speech/docs]",
    "ElevenLabs Streaming": "Low-latency streaming pricing page + product claims [source: https://elevenlabs.io/pricing]",
    "Azure Neural TTS": "Real-time TTS product family support [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]"
  },
  {
    "Metric": "Hindi/Kannada fit",
    "Qwen Omni TTS (Tina)": "Project prompts target Indian-language interaction [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "OpenAI TTS-1 / TTS-1-HD": "Multilingual family; exact Kannada voice count not captured [source: https://openai.com/index/introducing-the-realtime-api/]",
    "Google Cloud TTS WaveNet": "Broad voice/language family [source: https://cloud.google.com/text-to-speech/docs]",
    "ElevenLabs Streaming": "Broad TTS product suite [source: https://elevenlabs.io/pricing]",
    "Azure Neural TTS": "Hindi and Kannada locales/voices documented [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]"
  },
  {
    "Metric": "Separate API call required",
    "Qwen Omni TTS (Tina)": "No [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "OpenAI TTS-1 / TTS-1-HD": "Yes",
    "Google Cloud TTS WaveNet": "Yes",
    "ElevenLabs Streaming": "Yes",
    "Azure Neural TTS": "Yes"
  },
  {
    "Metric": "Cost note",
    "Qwen Omni TTS (Tina)": "Speech-to-speech token pricing inside omni model [source: https://docs.qwencloud.com/developer-guides/getting-started/pricing]",
    "OpenAI TTS-1 / TTS-1-HD": "See OpenAI audio/TTS pricing path via Realtime article [source: https://openai.com/index/introducing-the-realtime-api/]",
    "Google Cloud TTS WaveNet": "See public Cloud TTS pricing path [source: https://cloud.google.com/text-to-speech/docs]",
    "ElevenLabs Streaming": "Commercial pricing page with low-latency notes [source: https://elevenlabs.io/pricing]",
    "Azure Neural TTS": "See Azure Speech docs [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]"
  }
]

df = pd.DataFrame(comparison_rows)
display(df)


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels=["Qwen Omni", "OpenAI TTS", "Google TTS", "ElevenLabs", "Azure TTS"]
values=[1, 0, 0, 0, 0]
colors=["#4fc3f7", "#555555", "#555555", "#555555", "#555555"]
fig, ax = plt.subplots(figsize=(10,5))
setup_ax(ax, 'Ally Vision v2 — Integrated Speech Session Comparison')
ax.bar(labels, values, color=colors)
ax.set_ylabel('Integrated with same live session (1=yes)', color='white')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'category7_tts_voice_comparison_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
categories=["Integrated session", "Streaming output", "Separate API avoided"]
series={"Qwen Omni": {"values": [1, 1, 1], "color": "#4fc3f7"}, "OpenAI TTS": {"values": [0, 1, 0], "color": "#555555"}, "Google TTS": {"values": [0, 1, 0], "color": "#555555"}, "ElevenLabs": {"values": [0, 1, 0], "color": "#555555"}, "Azure TTS": {"values": [0, 1, 0], "color": "#555555"}}
x=np.arange(len(categories))
width=0.8/max(1,len(series))
fig, ax = plt.subplots(figsize=(12,5))
setup_ax(ax, 'Ally Vision v2 — TTS Workflow Comparison')
for idx, (label, spec) in enumerate(series.items()):
    offset=(idx-(len(series)-1)/2)*width
    ax.bar(x+offset, spec['values'], width=width, label=label, color=spec['color'])
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'category7_tts_voice_comparison_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source | URL | Accessed via |
|---|--------|-----|-------------|
| 1 | Qwen pricing | https://docs.qwencloud.com/developer-guides/getting-started/pricing | Tavily extract |
| 2 | OpenAI Realtime API | https://openai.com/index/introducing-the-realtime-api/ | direct public fetch |
| 3 | Google Cloud TTS docs | https://cloud.google.com/text-to-speech/docs | direct public fetch |
| 4 | Azure Speech support | https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt | direct public fetch |
| 5 | ElevenLabs pricing | https://elevenlabs.io/pricing | direct public fetch |
| 6 | Ally realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | project code URL |


## CONCLUSION

Qwen Omni wins because Ally Vision v2 does not have to break one response into two paid/latency-bearing phases. Speech output is part of the same realtime session, which reduces orchestration overhead and keeps interruption handling simple.

→ Chosen for Ally Vision v2 ✅
